## Dataset Token and Atom Stats

This notebook is a comparison across the training and fine-tuning datasets used in this project. The goal is to measure how large the CIFs and unit cells are so the token budget and structural complexity of each dataset can be compared side by side.

#### Making the dataframes with token count labels for each CIF

In [ ]:
import __init__

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/jarvis-XRD-processing-2/jarvis-XRD-unproc.parquet \
    --output_parquet HF-databases/jarvis-XRD-processing-2/jarvis-XRD-count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/mp_20_pxrd/mp_20_pxrd.parquet \
    --output_parquet HF-databases/mp_20_pxrd/mp_20_count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/mpdb-slme/mpdb-slme.parquet \
    --output_parquet HF-databases/mpdb-slme/mpdb-slme-count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/mattergen_dev/mattergen_den_ehull_dedup.parquet \
    --output_parquet HF-databases/mattergen_dev/mattergen_den_ehull_count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/mattergen_dev/mattergen_xrd_ehull_dedup.parquet \
    --output_parquet HF-databases/mattergen_dev/mattergen_xrd_ehull_count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/mpdb_processing/mpdb_2prop_data.parquet \
    --output_parquet HF-databases/mpdb_processing/mpdb_2prop_count.parquet \
    --num_workers 32 \
    --count_tokens

In [ ]:
!python _utils/_preprocessing/_cleaning.py \
    --input_parquet HF-databases/chili100k/chili100k_dedup_leakproof.parquet \
    --output_parquet HF-databases/chili100k/chili100k_count.parquet \
    --num_workers 32 \
    --count_tokens

### Load datasets

In [ ]:
import __init__
from _utils._notebook_utils.y_dataset_stats_utils import compute_atom_counts, load_count_datasets, plot_dataset_stats

In [ ]:
# (name, parquet_path, context_token_limit)
DATASETS = [
    ("Jarvis XRD", "HF-databases/jarvis-XRD-processing-2/jarvis-XRD-count.parquet", 1024),
    ("MP-20 XRD", "HF-databases/mp_20_pxrd/mp_20_count.parquet", 1024),
    ("MP SLME", "HF-databases/mpdb-slme/mpdb-slme-count.parquet", 1024),
    ("Mattergen Alex-MP-20", "HF-databases/mattergen_dev/mattergen_den_ehull_count.parquet", 1024),
    ("Mattergen XRD", "HF-databases/mattergen_dev/mattergen_xrd_ehull_count.parquet", 1024),
    ("MP Bandgap", "HF-databases/mpdb_processing/mpdb_2prop_count.parquet", 1024),
    ("CHILI-100K", "HF-databases/chili100k/chili100k_count.parquet", 1536),
]

loaded = load_count_datasets(DATASETS)

### Add atom counts and summarize

The next cell parses the CIF text to recover both conventional and reduced cell atom counts. We do that because most diffusion models use Niggli-reduced cells for training so it serves as a comparative measure. If the columns are already in the parquet, skipped.

In [ ]:
loaded = compute_atom_counts(loaded, num_workers=32)

#### Summary

In [ ]:
import pandas as pd

summary_rows = []
for name, df, ctx in loaded:
    if df is None:
        continue

    row = {
        "Dataset": name,
        "Structures": len(df),
        "Context limit": ctx,
        "% beyond context": 100 * (df["token_count"] > ctx).mean(),
        "Token median": df["token_count"].median(),
        "Token p95": df["token_count"].quantile(0.95),
    }

    if "conv_count" in df.columns and "prim_count" in df.columns:
        row.update(
            {
                "Conv atoms median": df["conv_count"].median(),
                "Conv atoms p95": df["conv_count"].quantile(0.95),
                "Prim atoms median": df["prim_count"].median(),
                "Prim atoms p95": df["prim_count"].quantile(0.95),
            }
        )

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).round(2).set_index("Dataset")
summary_df

#### Optionally Plot

In [ ]:
plot_dataset_stats(loaded)